In [ ]:
!pip install ollama tqdm

In [1]:
import json
import os
import time
import hashlib
from tqdm import tqdm
import ollama

In [2]:
# Load cleaned corpus
corpus_path = r"C:\Users\MIND & MATTER\Desktop\Qwen_CustomDomain\Dataset\java_corpus.txt"

with open(corpus_path, "r", encoding="utf-8") as f:
    cleaned = f.read()

def chunk_text(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

chunks = chunk_text(cleaned)
print(f"Total chunks loaded: {len(chunks)}")

Total chunks loaded: 6717


In [3]:
output_path       = r"C:\Users\MIND & MATTER\Desktop\Qwen_CustomDomain\Dataset\qa_pairs.jsonl"
chat_output_path  = r"C:\Users\MIND & MATTER\Desktop\Qwen_CustomDomain\Dataset\qa_pairs_chat.jsonl"
progress_path     = r"C:\Users\MIND & MATTER\Desktop\Qwen_CustomDomain\Dataset\progress.json"

MODEL         = "qwen2.5:7b"
SAVE_EVERY    = 50
SLEEP_BETWEEN = 0.5

In [4]:
def generate_qa(chunk, chunk_id):
    prompt = f"""You are a Java expert creating a high-quality fine-tuning dataset.

Given the Java tutorial text below, generate exactly 4 QA pairs:
- 1 "what" question (factual)
- 1 "how" question (process/mechanism)
- 1 "why" question (reasoning/benefit)
- 1 unanswerable question (topic NOT covered in the text)

Text:
{chunk}

Strict Rules:
1. Answerable questions MUST be answerable ONLY from the text above — no outside knowledge.
2. Answers must be COMPLETE — never cut off mid-sentence.
3. The "answerable: false" entry MUST have answer: "I don't have enough information in my knowledge base to answer this."
4. The "answerable: true" entries MUST have the actual answer text, never the refusal string.
5. Include the context field (copy the relevant sentence(s) from the text that support the answer).
6. Return ONLY a valid JSON array — no explanation, no markdown, no code fences.

Format:
[
  {{"question": "...", "answer": "...", "context": "...", "answerable": true, "chunk_id": {chunk_id}}},
  {{"question": "...", "answer": "...", "context": "...", "answerable": true, "chunk_id": {chunk_id}}},
  {{"question": "...", "answer": "...", "context": "...", "answerable": true, "chunk_id": {chunk_id}}},
  {{"question": "...", "answer": "I don't have enough information in my knowledge base to answer this.", "context": "", "answerable": false, "chunk_id": {chunk_id}}}
]"""

    for attempt in range(3):
        try:
            response = ollama.chat(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                options={"temperature": 0.3, "num_predict": 1024}
            )

            raw = response["message"]["content"].strip()
            raw = raw.replace("```json", "").replace("```", "").strip()

            start = raw.find("[")
            end   = raw.rfind("]") + 1
            if start == -1 or end == 0:
                raise ValueError("No JSON array found")

            raw      = raw[start:end]
            qa_pairs = json.loads(raw)

            # ── Validation & Auto-fix ──────────────────────────────────────
            validated = []
            REFUSAL   = "I don't have enough information in my knowledge base to answer this."

            for qa in qa_pairs:
                # Fix: answerable=true but answer is the refusal string → skip
                if qa.get("answerable") is True and qa.get("answer", "").strip() == REFUSAL:
                    print(f"\n⚠️  chunk {chunk_id}: auto-fixed wrong answerable=true+refusal answer — skipped")
                    continue

                # Fix: answerable=false but answer is NOT the refusal string → correct it
                if qa.get("answerable") is False and qa.get("answer", "").strip() != REFUSAL:
                    qa["answer"] = REFUSAL

                # Fix: ensure context field exists
                if "context" not in qa:
                    qa["context"] = ""

                validated.append(qa)

            return validated

        except Exception as e:
            print(f"\n⚠️ Attempt {attempt+1} failed for chunk {chunk_id}: {e}")
            time.sleep(1)

    return []

In [5]:
# Tracks seen question hashes across the full run (in-memory)
seen_questions = set()

def _question_hash(question: str) -> str:
    return hashlib.md5(question.strip().lower().encode()).hexdigest()

def load_existing_hashes():
    """Populate seen_questions from an already-existing output file (for resume)."""
    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    item = json.loads(line)
                    seen_questions.add(_question_hash(item["question"]))
                except Exception:
                    pass
    print(f"Loaded {len(seen_questions)} existing question hashes for dedup.")

def to_chat_format(qa: dict) -> dict:
    """Convert a raw QA pair to OpenAI chat fine-tuning format."""
    system_msg = "You are a helpful Java programming assistant. Answer questions based only on the provided context."
    if qa.get("answerable") and qa.get("context"):
        user_content = f"Context: {qa['context']}\n\nQuestion: {qa['question']}"
    else:
        user_content = f"Question: {qa['question']}"
    return {
        "messages": [
            {"role": "system",    "content": system_msg},
            {"role": "user",      "content": user_content},
            {"role": "assistant", "content": qa["answer"]}
        ]
    }

def append_qa(qa_pairs):
    with open(output_path, "a", encoding="utf-8") as f_raw, \
         open(chat_output_path, "a", encoding="utf-8") as f_chat:
        for qa in qa_pairs:
            h = _question_hash(qa["question"])
            if h in seen_questions:
                print(f"\n🔁 Duplicate skipped: {qa['question'][:60]}...")
                continue
            seen_questions.add(h)
            f_raw.write(json.dumps(qa) + "\n")
            f_chat.write(json.dumps(to_chat_format(qa)) + "\n")

def save_progress(chunk_index):
    with open(progress_path, "w") as f:
        json.dump({"last_chunk": chunk_index}, f)

In [6]:
def load_progress():
    if os.path.exists(progress_path):
        with open(progress_path, "r") as f:
            data = json.load(f)
        print(f"Resuming from chunk {data['last_chunk']} of {len(chunks)}")
        return data["last_chunk"]
    else:
        print("No previous progress found. Starting fresh.")
        return 0

load_existing_hashes()   # ← load dedup hashes before starting
start_from = load_progress()

Loaded 1548 existing question hashes for dedup.
Resuming from chunk 450 of 6717


In [ ]:
failed_chunks = []

for i in tqdm(range(start_from, len(chunks)), desc="Generating QA pairs"):
    try:
        qa_pairs = generate_qa(chunks[i], chunk_id=i)
        append_qa(qa_pairs)

        # Save progress every 50 chunks
        if i % SAVE_EVERY == 0:
            save_progress(i)
            print(f"\n✅ Progress saved at chunk {i}/{len(chunks)}")

        time.sleep(SLEEP_BETWEEN)

    except Exception as e:
        print(f"\n⚠️ Failed chunk {i}: {e}")
        failed_chunks.append(i)
        continue

# Final save
save_progress(len(chunks))
print(f"\n🎉 Done. Failed chunks: {failed_chunks}")

In [ ]:
def print_stats(path, label):
    if not os.path.exists(path):
        print(f"{label}: file not found")
        return
    total = answerable = unanswerable = 0
    q_types = {"what": 0, "how": 0, "why": 0, "other": 0}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            # chat format has "messages", raw format has "question"
            if "messages" in item:
                total += 1
                continue
            total += 1
            if item["answerable"]:
                answerable += 1
                q = item["question"].lower()
                if q.startswith("what"):   q_types["what"]  += 1
                elif q.startswith("how"):  q_types["how"]   += 1
                elif q.startswith("why"):  q_types["why"]   += 1
                else:                      q_types["other"] += 1
            else:
                unanswerable += 1
    print(f"\n── {label} ──")
    print(f"  Total QA pairs   : {total}")
    print(f"  Answerable       : {answerable}")
    print(f"  Unanswerable     : {unanswerable}")
    if answerable:
        print(f"  Question types   : {q_types}")
    print(f"  Chunks processed : ~{total // 4}")
    print(f"  Chunks remaining : ~{len(chunks) - total // 4}")

print_stats(output_path,      "Raw JSONL (with context)")
print_stats(chat_output_path, "Chat format JSONL (for fine-tuning)")

In [ ]:
# ── RESUME CELL ──
# Run this cell only when resuming after a break
# It reads progress.json and restarts from where you stopped

start_from = load_progress()
failed_chunks = []

for i in tqdm(range(start_from, len(chunks)), desc="Resuming QA generation"):
    try:
        qa_pairs = generate_qa(chunks[i], chunk_id=i)
        append_qa(qa_pairs)

        if i % SAVE_EVERY == 0:
            save_progress(i)
            print(f"\n✅ Progress saved at chunk {i}/{len(chunks)}")

        time.sleep(SLEEP_BETWEEN)

    except Exception as e:
        print(f"\n⚠️ Failed chunk {i}: {e}")
        failed_chunks.append(i)
        continue

save_progress(len(chunks))
print(f"\n🎉 Done. Failed chunks: {failed_chunks}")

Resuming from chunk 450 of 6717


Resuming QA generation:   0%|                                                                 | 0/6267 [00:00<?, ?it/s]


🔁 Duplicate skipped: Why would an if-else statement be used in this context?...

🔁 Duplicate skipped: What is the purpose of the variable 'principal' in this code...

✅ Progress saved at chunk 450/6717


Resuming QA generation:   0%|                                                     | 11/6267 [01:36<13:09:58,  7.58s/it]


🔁 Duplicate skipped: What does the algorithm require us to do first?...


...

Resuming QA generation:  44%|█████████████████████▉                            | 2750/6267 [6:24:16<7:17:38,  7.47s/it]


✅ Progress saved at chunk 3200/6717


Resuming QA generation:  44%|██████████████████████                            | 2766/6267 [6:26:09<7:12:34,  7.41s/it]


🔁 Duplicate skipped: What does the book cover?...


Resuming QA generation:  44%|██████████████████████                            | 2770/6267 [6:26:34<6:11:57,  6.38s/it]

In [ ]:
# Run this anytime to see how many QA pairs generated so far
count = 0
with open(output_path, "r", encoding="utf-8") as f:
    for line in f:
        count += 1

answerable   = 0
unanswerable = 0
with open(output_path, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        if item["answerable"]:
            answerable += 1
        else:
            unanswerable += 1

print(f"Total QA pairs      : {count}")
print(f"Answerable          : {answerable}")
print(f"Unanswerable        : {unanswerable}")
print(f"Chunks processed    : {count // 4}")
print(f"Chunks remaining    : {len(chunks) - count // 4}")